In [3]:
import numpy as np

# test = np.linspace(-8.93239e-9, -4e-8, num=20) 
# print(test)

lam5_test = np.linspace(-8.93239e-9, 1e-9, num=49) 
lam5_test = np.sort(np.append(lam5_test, 0.0))#now 20

print(lam5_test)

[-8.93239000e-09 -8.72546521e-09 -8.51854042e-09 -8.31161563e-09
 -8.10469083e-09 -7.89776604e-09 -7.69084125e-09 -7.48391646e-09
 -7.27699167e-09 -7.07006688e-09 -6.86314208e-09 -6.65621729e-09
 -6.44929250e-09 -6.24236771e-09 -6.03544292e-09 -5.82851813e-09
 -5.62159333e-09 -5.41466854e-09 -5.20774375e-09 -5.00081896e-09
 -4.79389417e-09 -4.58696937e-09 -4.38004458e-09 -4.17311979e-09
 -3.96619500e-09 -3.75927021e-09 -3.55234542e-09 -3.34542063e-09
 -3.13849583e-09 -2.93157104e-09 -2.72464625e-09 -2.51772146e-09
 -2.31079667e-09 -2.10387188e-09 -1.89694708e-09 -1.69002229e-09
 -1.48309750e-09 -1.27617271e-09 -1.06924792e-09 -8.62323125e-10
 -6.55398333e-10 -4.48473542e-10 -2.41548750e-10 -3.46239583e-11
  0.00000000e+00  1.72300833e-10  3.79225625e-10  5.86150417e-10
  7.93075208e-10  1.00000000e-09]


In [1]:
# Core imports man
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import time
import pyoscode

# Numerical libraries
from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
from scipy.interpolate import (
    UnivariateSpline, splrep, splev, CubicSpline, 
    interp1d, PchipInterpolator, InterpolatedUnivariateSpline
)

import numdifftools as nd

# Random number generator (GSL)
import pygsl.rng

# custom InflationModels code to path the one below is for wkb approximation method
sys.path.append(
    '/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels'
)

#the path below assumes the original numerics from Deyan's code
# sys.path.append('/Users/epmeador/Desktop/InflationModels-master')

# Enable spectra
SPECTRUM = True

# Local modules from InflationModels
from MacroDefinitions import *
from calcpath import *
from int_de import *
if SPECTRUM:
#     from spectrum_noS0_fixedNstar import * #this is mode modified one that I have been working with for wkb approx
#     from spectrum_OG import * #this is the original spectrum from full numerical code
#     from spectrum_pyoscode_tensor_clean import * #this is the spectrum from pyoscode 
#     from spectrum_pyoscode_scalar import * #this is the spectrum from pyoscode with scalar 
#     from spectrum_pyoscode_optimized_wkb import *
#     from spectrum_pyoscode_test_N import *
    from spectrum_OG_nanoscale_nodiagnostics import * #this is equivalent to the original spectrum from full numerical code w/o diagnostics

# ========================
# GLOBAL SETTINGS
# ========================
NEQS = 8  # Updated from 7 to 8
SPECTRUM = True
SAVEPATHS = True

NMAX = 1.2
NMIN = 0.3

#if I want to expand the lam5 space for higgs values I can do a symmetric expansion around what I know works
LAM5_MIN = -3e-8   # ~3-4x more negative than base
LAM5_MAX = 2.0e-8    # 2-3x current upper bound
LAM5_BASE = -8.92461e-9
NUM_LAM5_GRID = 398  # Updated from NUM_LAM4_GRID to NUM_LAM5_GRID
lam5_set = np.linspace(LAM5_MIN, LAM5_MAX, num=NUM_LAM5_GRID)
lam5_set = np.sort(np.append(lam5_set, [LAM5_BASE, 0.0]))

print(f"Total models: {len(lam5_set)}")  # Should be 400 (or 399-400)
print(f"Base λ₅={LAM5_BASE:.6e} included: {LAM5_BASE in lam5_set}")
print(f"Zero included: {0.0 in lam5_set}")

#This sets how many nontrivial models you are running (interesting models you want)
NUMPOINTS = 1        # number of nontrivial models per λ5

#This will give us the min and max number of e-folds we are looking for
NUMEFOLDSMAX = 65.0
NUMEFOLDSMIN = 57.0

# #Here we are writing out our output files we can name to analyze data
BASE_OUTDIR = "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8"  # Updated from neqs7 to neqs8

# RNG initialization process
#This can be used to generate a random generator that works well with simulations
my_random = pygsl.rng.ranlxd2()
my_random.set(0)
np.random.seed(0)

# ========================
# Support Functions
# ========================

#Here we define a class for several variables, this will initialize several variables
#The size we are setting for Y and initY is the same as NEQs which may be number of equations and is set in flow.py
#We set a state, the initial state, an empty string in the class, number of points, and e-folds
class Calc:
    def __init__(self):
        self.Y = np.zeros(NEQS, dtype=float, order='C')
        self.initY = np.zeros(NEQS, dtype=float, order='C')
        self.ret = ""
        self.npoints = 0
        self.Nefolds = 0.0

#Below we are initializing our starting slow roll parameter values
#We should be able to choose what ell we are extending to.
def pick_init_vals(lam5):  # Updated from lam4 to lam5

    #This is what is making it slow I think, uncomment init_vals below
   # current higgs values
    init_vals = np.zeros(NEQS, dtype=float, order='C')
    init_vals[0] = 5.5 #phi0 #i dont think this really affects it??
    init_vals[1] = 1.0 #H0
    init_vals[2] = 0.000209237 #epsilon0
    init_vals[3] = -0.0342419 #sigma0
    init_vals[4] = 0.000278972#2lam0
    init_vals[5] = -4.60971e-06
    init_vals[6] = 6.87065e-08
    init_vals[7] = lam5


    
#     #Below is the original set of values used in your code
#         #original values 
#     init_vals = np.zeros(NEQS, dtype=float, order='C')
#     init_vals[0] = 0.0
#     init_vals[1] = 1.0
#     init_vals[2] = my_random.uniform() * 0.8
#     init_vals[3] = my_random.uniform() - 0.5
#     init_vals[4] = my_random.uniform()*0.1 - 0.05

#     prefact = 0.05
    
#     for i in range (5 , NEQS):
#         init_vals[i] = my_random.uniform() * prefact - (0.5*prefact)
#         prefact *= 0.1
       
    init_Nefolds = my_random.uniform() * (NUMEFOLDSMAX - NUMEFOLDSMIN) + NUMEFOLDSMIN
    return init_vals, init_Nefolds

#SAVING PATHS
#This next part of the code will print either asymptote or nontrivial 
#And decides when to save the code 
#This is based on what the model itself is doing

#This function below will calculation the spectrum computed from y
#As long as its in a particular range
#This is where I would limit my range of spectrum models of interest
#Otherwise it returns false
def we_should_calc_spec(y):
    return (specindex(y) > NMIN and specindex(y) < NMAX)

#Specifically, we will save a model with some interesting dynamics
#And this means either asymptote or nontrivial - for us nontrivial 
#Also only if the path has not been saved yet, and only for 
#Every nth successful model
#This governs path saving in the non-spectral case
def we_should_save_path(retval, save, pointcount, printevery):
    return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)

#Overall, this writes model trajectory to a file
#the SRP at each integration step, the number of N remaining, gives a reconstructed V, and e_H
# Open output file
def save_path(y, N, kount, fname):
    with open(fname, "w") as outfile:
    # Output intermediate data from the integration
        for i in range(kount):
            for j in range(NEQS):
            #get y variable as a function of i in kount
            #should be like SRP as a function of time
            #j loops over NEQS which are used 
                outfile.write("%le " % y[j, i])
            outfile.write("%lf " % N[i])
            #Will also write out N at that time step
#             V = 3 * y[1, i]**2 * (1. - y[2, i]/3.) #this one is wrong, i think this is what he had
            V = (3./(8.*np.pi)) * y[1, i] * y[1, i] * (1.-y[2, i]/3.) #what the original code had
            outfile.write("%le %le\n" % (V, (V*y[2, i])/(3. - y[2, i]))) #I think this is KE

# ========================
# Main Loop
# ========================
def run_neqs8_models():  # Updated from run_neqs7_models to run_neqs8_models
    summary_records = []
    
    for lam5 in lam5_set:  # Updated from lam4 to lam5
        print(f"\n=== Running λ5 = {lam5:.1e} ===")

        # Output directories
        OUTDIR = f"{BASE_OUTDIR}/lam5_{lam5:.1e}" # Updated from lam4 to lam5
        os.makedirs(OUTDIR, exist_ok=True)
        OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
        OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

        # Open output files for writing
        try:
            outfile1 = open(OUTFILE1_NAME, "w")
            outfile2 = open(OUTFILE2_NAME, "w")
        except IOError as e:
            print("Could not open output files: ", e)
            sys.exit()

        # Spectrum arrays
        if SPECTRUM:
            #Scalar mode function
            #u_s = np.empty((2, knos))
            u_s = np.empty((2, knos))
            #Tensor mode function
            #u_t = np.empty((2, knos))
            u_t = np.empty((2, knos))
            #This is an empty array of size NEQS + 1
            #This can be used to store slow roll variables plus some extra variable, maybe N 
            y_final = np.empty(NEQS + 1)
            #if i want to save raw power spectra as well
#             u_s_raw = np.empty((2, knos))
#             u_t_raw = np.empty((2, knos))
            #Below will track how many spectra have been saved
            spec_count = 0

        # Initialize counters and allocate buffers(?)
        #sets everything to 0
        # iters = total number of iterations
        calc = Calc()
        iters = 0
        points = 0
        outcount = 0
        asymcount = 0
        nontrivcount = 0
        insuffcount = 0
        noconvcount = 0
        badncount = 0
        errcount = 0
        savedone = 0

        # Trial loop
        #Main part of MC loop: tries random models and decides whether to keep or discard them
        #This will keep looping as long as nontrivcount is less than NUMPOINTS
        #So it will find random inflation models until I have found enough nontrivial ones
        #nontrivial is an interesting model, if valid it will increment nontrivcount until you get NUMPOINTS
        #loop will stop once you get NUMPOINTS OR hit iters>200
        while nontrivcount < NUMPOINTS:
            iters += 1
            if iters > 200:  # failsafe
                break

            #this will print progress updates
            #every 100 iterations do something, every 1000 print status report
            #if multiple of 100 not 1000 print .
            if iters % 100 == 0:
                print(f"  Iter {iters}, nontriv={nontrivcount}")

            # pick initial conditions right now it is not random
            yinit, calc.Nefolds = pick_init_vals(lam5)  # Updated from lam4 to lam5
            #creates copy of random initial conditions created by pick_init_vals
            y = yinit.copy()

            # state arrays
            path = np.array([[]])
            N = np.array([])

            # integrate flow equations
            #this will call the calcpath function using random fluc copy, 
            #path from srp evolution, N, and calc 
            # calc stores state! So a string that categorizes this
            t0 = time.perf_counter()
            calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
            t1 = time.perf_counter()
            print(f"calcpath runtime: {t1 - t0:.4f} s")
            print(f"  -> {calc.ret}")
            
            #Confirm diagnostic stuff
            print("\n=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===")

            # print initial slow-roll params
            print(f"Initial values (yinit): {yinit}")
            print(f"  φ0 = {yinit[0]:.6e}")
            print(f"  H0 = {yinit[1]:.6e}")
            print(f"  ε0 = {yinit[2]:.6e}")
            print(f"  σ0 = {yinit[3]:.6e}")
            print(f"  λ₂ = {yinit[4]:.6e}")
            if len(yinit) > 5:
                print(f"  λ₅ = {yinit[5]:.6e}")  # Updated from λ₄ to λ₅
            print(f"Initial Nefolds = {calc.Nefolds:.3f}")

            # check integrator tolerance (read from pygsl object if available)
            try:
                import inspect
                import pygsl.odeiv as odeiv
                s = odeiv.step_rk4(len(yinit), derivs1)
                c = odeiv.control_y_new(s, 1e-8, 1e-8)  # test new control to show expected
                print(f"GSL integrator class: {s.__class__.__name__}")
                print(f"Expected tolerances: atol=1e-8, rtol=1e-8")
            except Exception as e:
                print("Could not check GSL integrator:", e)

            print("===============================================\n")

            # handle outcome - i told it to get rid of asymptote
            #the original code kept them for late time attractors
            #if it is an asymptote we  proceed to find spectral indices and write them to a file
            #this is done after checking to see if specindex is between NMIN and NMAX
            #this is our nr file
            #this asymptote approached a late time attracror, could give valid observables, have to manually screen
            #mathematically consistent, exact solutions to flow,  usefulf or low-scale inflation models, can correspond to eternal inflation
            #they dont really allow for reheating which is why i dont like them
            #removed asymptote
            if calc.ret == "asymptote":
                asymcount += 1
                if asymcount > 100:
                    print("Too many asymptotes, stopping")
                    break
                continue

            #This block runs if nontrivial, which means it ended inflation after calc.Nefolds, not just asymptote
            #that block continues below
            if calc.ret == "nontrivial":
                # write observables
                r = tsratio(y)
                ns = specindex(y)
                alpha_s = dspecindex(y)
                outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")
                outfile1.flush()

                # write final srp parameters and appends number
                for i in range(NEQS):
                    outfile2.write("%le " % y[i])
                outfile2.write("%f\n" % calc.Nefolds)
                outfile2.flush()

                points += 1 #total valid models saved so far
                savedone = 0 #marks that havent saved in teh trajectory files yet
                nontrivcount += 1 #counts how many nontrivial models we've accepted

                # calculate spectra if needed
                if SPECTRUM and we_should_calc_spec(y):
                    print(f"  ns = {specindex(y):.3f}")
                    #this tells us what spectrum we are evaluating
                    print(f"  -> Evaluating spectrum {spec_count}")
                    #we will need to initalize ps evolution using slow roll values
                    #time step 3 so that its early enough in inflation so most modes are w/n horizon
                    y_final[:NEQS] = path[:NEQS, 3] #this grabs all slow roll parameters at time step 3
                    y_final[NEQS] = N[3] #this add the value of N at that time step
                    #dont forget : means slicing, path[:NEQS, 3] = path[0:NEQS-1,3]
                    #which is rows 0-NEQS-1, at column 3
                    
                    t0 = time.perf_counter()
                    spectrum_status = spectrum(y_final, y, u_s, u_t, calc.Nefolds,
                                               derivs1, scalarsys, tensorsys)
# #if i wanna save the raw power spectrum activate the line below
#                     spectrum_status = spectrum(y_final, y, u_s, u_t, calc.Nefolds,
#                                                derivs1, scalarsys, tensorsys,u_s_raw, u_t_raw)
                    
                    t1 = time.perf_counter()
                    print(f"spectrum runtime: {t1 - t0:.4f} s")

#                     #runs through spectrum module which uses that prepared final state
#                     spectrum_status = spectrum(y_final, y, u_s, u_t, calc.Nefolds,
#                                                derivs1, scalarsys, tensorsys)
                    if spectrum_status:
                        errcount += 1
                    #if something is wrong it will start counting errors

                    # save spectra
                    spec_s_name = f"{OUTDIR}/spec_s{spec_count:03d}_neqs{NEQS}.dat"
                    spec_t_name = f"{OUTDIR}/spec_t{spec_count:03d}_neqs{NEQS}.dat"
                    np.savetxt(spec_s_name, u_s[:, :knos].T)
                    np.savetxt(spec_t_name, u_t[:, :knos].T)
                    
#                     #if i wanna save raw spectra
#                     spec_s_raw_name = f"{OUTDIR}/spec_s_raw{spec_count:03d}_neqs{NEQS}.dat"
#                     spec_t_raw_name = f"{OUTDIR}/spec_t_raw{spec_count:03d}_neqs{NEQS}.dat"

#                     np.savetxt(spec_s_raw_name, u_s_raw[:, :knos].T)
#                     np.savetxt(spec_t_raw_name, u_t_raw[:, :knos].T)
                    spec_count += 1

                    #Remember we are only computing power spectra for nontrivial models.
                    #For these models, inflation ends.
                    #To compute power spectrum you evolve mode k through horizon crossing k=aH
                    #And you evaluate perturbation amplitudes after horizon exit
                    #Attractor models never exit inflation

                #added a line in calcpath
                #normalize path if spectrum=True
                if SPECTRUM:
                    for j in range(calc.npoints):
# #                         #print("Final Hubble y[1] =", y[1])
# #                         #this normalization shifts all phi vals
#                         #it subtracts the final phi from each one so last pt is 0
#                         #phi is recentered so end of inflation is at phi=0
#                         path[0, j] -= path[0, calc.npoints-1]
#                         #we then rescale the hubble values to get correct scale
#                         #if y[1] is less than zero this makes all H negative 
# normalize path if spectrum=True (match original)
                        path[0, :calc.npoints] -= path[0, calc.npoints-1]
                        path[1, :calc.npoints] *= y[1]
#                         path[1, j] = path[1, j] * abs(y[1]) #original line 
#                         path[1, j] *= abs(y[1])
                        
#                 #Added this        
#                 Hnorm = 0.00001 * 2 * np.pi * np.sqrt(y[2]) / y[1]
#                 path[1, :] *= Hnorm

                # save path
                path_name = f"{OUTDIR}/path_neqs{NEQS}_lam5{lam5:.1e}_{outcount:03d}.dat"  # Updated from lam4 to lam5
                print(f"  -> Saving path {path_name}")
                save_path(path, N, calc.npoints, path_name)
                outcount += 1

                # add summary record?
                summary_records.append({
                    "lam5": lam5, "r": r, "n_s": ns,
                    "alpha_s": alpha_s, "Nefolds": calc.Nefolds
                })

            elif calc.ret == "insuff":
                insuffcount += 1
            elif calc.ret == "noconverge":
                noconvcount += 1
            else:
                errcount += 1

        # close output files
        outfile1.close()
        outfile2.close()

    # write summary CSV
    summary_df = pd.DataFrame(summary_records)
    summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
    summary_df.to_csv(summary_file, index=False)
    print(f"\nSummary written to {summary_file}")

# if __name__ == "__main__":
#     run_neqs8_models()  # Updated from run_neqs7_models to run_neqs8_models

print('env', sys.executable)

import platform, numpy, scipy, pandas, matplotlib
print(f"Python: {platform.python_version()}")
print(f"CPU Architecture: {platform.machine()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

%time run_neqs8_models()  # Updated from run_neqs7_models to run_neqs8_models


ks_eval_extend: 7.791356504680627e-06 to 646999999999999.0 N = 1575
ks_extend:      7.791356504680627e-06 to 646999999999999.0 N = 214

=== Running λ5 = -8.9e-09 ===
ε crosses 1 at step 117
Final λ₄ max: 1.0226304727430984
  -> nontrivial
  -> Evaluating spectrum 0
Starting for this mode
0
[DEBUG] scalar mode 0 took 0.149 s
[DEBUG] tensor mode 0 took 0.138 s
Mode 1/214 (k = 4.215e-63) took 0.29 s
Finished for this mode

Starting for this mode
1
[DEBUG] scalar mode 1 took 0.153 s
[DEBUG] tensor mode 1 took 0.139 s
Mode 2/214 (k = 5.228e-63) took 0.29 s
Finished for this mode

Starting for this mode
2
[DEBUG] scalar mode 2 took 0.151 s
[DEBUG] tensor mode 2 took 0.139 s
Mode 3/214 (k = 6.484e-63) took 0.29 s
Finished for this mode

Starting for this mode
3
[DEBUG] scalar mode 3 took 0.152 s
[DEBUG] tensor mode 3 took 0.137 s
Mode 4/214 (k = 8.042e-63) took 0.29 s
Finished for this mode

Starting for this mode
4
[DEBUG] scalar mode 4 took 0.151 s
[DEBUG] tensor mode 4 took 0.135 s
Mode 5/

[DEBUG] scalar mode 50 took 0.141 s
[DEBUG] tensor mode 50 took 0.127 s
Mode 51/214 (k = 1.998e-58) took 0.27 s
Finished for this mode

Starting for this mode
51
[DEBUG] scalar mode 51 took 0.140 s
[DEBUG] tensor mode 51 took 0.122 s
Mode 52/214 (k = 2.479e-58) took 0.26 s
Finished for this mode

Starting for this mode
52
[DEBUG] scalar mode 52 took 0.143 s
[DEBUG] tensor mode 52 took 0.127 s
Mode 53/214 (k = 3.074e-58) took 0.27 s
Finished for this mode

Starting for this mode
53
[DEBUG] scalar mode 53 took 0.139 s
[DEBUG] tensor mode 53 took 0.122 s
Mode 54/214 (k = 3.813e-58) took 0.26 s
Finished for this mode

Starting for this mode
54
[DEBUG] scalar mode 54 took 0.142 s
[DEBUG] tensor mode 54 took 0.124 s
Mode 55/214 (k = 4.729e-58) took 0.27 s
Finished for this mode

Starting for this mode
55
[DEBUG] scalar mode 55 took 0.136 s
[DEBUG] tensor mode 55 took 0.120 s
Mode 56/214 (k = 5.865e-58) took 0.26 s
Finished for this mode

Starting for this mode
56
[DEBUG] scalar mode 56 took 

[DEBUG] scalar mode 101 took 0.129 s
[DEBUG] tensor mode 101 took 0.108 s
Mode 102/214 (k = 1.175e-53) took 0.24 s
Finished for this mode

Starting for this mode
102
[DEBUG] scalar mode 102 took 0.130 s
[DEBUG] tensor mode 102 took 0.115 s
Mode 103/214 (k = 1.458e-53) took 0.25 s
Finished for this mode

Starting for this mode
103
[DEBUG] scalar mode 103 took 0.127 s
[DEBUG] tensor mode 103 took 0.115 s
Mode 104/214 (k = 1.808e-53) took 0.24 s
Finished for this mode

Starting for this mode
104
[DEBUG] scalar mode 104 took 0.129 s
[DEBUG] tensor mode 104 took 0.113 s
Mode 105/214 (k = 2.242e-53) took 0.24 s
Finished for this mode

Starting for this mode
105
[DEBUG] scalar mode 105 took 0.129 s
[DEBUG] tensor mode 105 took 0.112 s
Mode 106/214 (k = 2.781e-53) took 0.24 s
Finished for this mode

Starting for this mode
106
[DEBUG] scalar mode 106 took 0.126 s
[DEBUG] tensor mode 106 took 0.108 s
Mode 107/214 (k = 3.449e-53) took 0.23 s
Finished for this mode

Starting for this mode
107
[DEB

[DEBUG] tensor mode 152 took 0.109 s
Mode 153/214 (k = 6.911e-49) took 0.24 s
Finished for this mode

Starting for this mode
153
[DEBUG] scalar mode 153 took 0.125 s
[DEBUG] tensor mode 153 took 0.107 s
Mode 154/214 (k = 8.571e-49) took 0.23 s
Finished for this mode

Starting for this mode
154
[DEBUG] scalar mode 154 took 0.125 s
[DEBUG] tensor mode 154 took 0.105 s
Mode 155/214 (k = 1.063e-48) took 0.23 s
Finished for this mode

Starting for this mode
155
[DEBUG] scalar mode 155 took 0.125 s
[DEBUG] tensor mode 155 took 0.108 s
Mode 156/214 (k = 1.318e-48) took 0.23 s
Finished for this mode

Starting for this mode
156
[DEBUG] scalar mode 156 took 0.124 s
[DEBUG] tensor mode 156 took 0.106 s
Mode 157/214 (k = 1.635e-48) took 0.23 s
Finished for this mode

Starting for this mode
157
[DEBUG] scalar mode 157 took 0.124 s
[DEBUG] tensor mode 157 took 0.107 s
Mode 158/214 (k = 2.028e-48) took 0.23 s
Finished for this mode

Starting for this mode
158
[DEBUG] scalar mode 158 took 0.139 s
[DEB

[DEBUG] tensor mode 202 took 0.096 s
Mode 203/214 (k = 3.276e-44) took 0.21 s
Finished for this mode

Starting for this mode
203
[DEBUG] scalar mode 203 took 0.114 s
[DEBUG] tensor mode 203 took 0.096 s
Mode 204/214 (k = 4.064e-44) took 0.21 s
Finished for this mode

Starting for this mode
204
[DEBUG] scalar mode 204 took 0.126 s
[DEBUG] tensor mode 204 took 0.096 s
Mode 205/214 (k = 5.040e-44) took 0.22 s
Finished for this mode

Starting for this mode
205
[DEBUG] scalar mode 205 took 0.114 s
[DEBUG] tensor mode 205 took 0.096 s
Mode 206/214 (k = 6.251e-44) took 0.21 s
Finished for this mode

Starting for this mode
206
[DEBUG] scalar mode 206 took 0.115 s
[DEBUG] tensor mode 206 took 0.096 s
Mode 207/214 (k = 7.753e-44) took 0.21 s
Finished for this mode

Starting for this mode
207
[DEBUG] scalar mode 207 took 0.114 s
[DEBUG] tensor mode 207 took 0.094 s
Mode 208/214 (k = 9.616e-44) took 0.21 s
Finished for this mode

Starting for this mode
208
[DEBUG] scalar mode 208 took 0.118 s
[DEB

...

[DEBUG] scalar mode 181 took 0.124 s
[DEBUG] tensor mode 181 took 0.105 s
Mode 182/214 (k = 3.561e-46) took 0.23 s
Finished for this mode

Starting for this mode
182
[DEBUG] scalar mode 182 took 0.121 s
[DEBUG] tensor mode 182 took 0.104 s
Mode 183/214 (k = 4.416e-46) took 0.22 s
Finished for this mode

Starting for this mode
183
[DEBUG] scalar mode 183 took 0.122 s
[DEBUG] tensor mode 183 took 0.103 s
Mode 184/214 (k = 5.477e-46) took 0.23 s
Finished for this mode

Starting for this mode
184
[DEBUG] scalar mode 184 took 0.120 s
[DEBUG] tensor mode 184 took 0.104 s
Mode 185/214 (k = 6.793e-46) took 0.22 s
Finished for this mode

Starting for this mode
185
[DEBUG] scalar mode 185 took 0.123 s
[DEBUG] tensor mode 185 took 0.104 s
Mode 186/214 (k = 8.426e-46) took 0.23 s
Finished for this mode

Starting for this mode
186
[DEBUG] scalar mode 186 took 0.122 s
[DEBUG] tensor mode 186 took 0.110 s
Mode 187/214 (k = 1.045e-45) took 0.23 s
Finished for this mode

Starting for this mode
187
[DEB

[DEBUG] tensor mode 14 took 0.146 s
Mode 15/214 (k = 8.591e-62) took 0.31 s
Finished for this mode

Starting for this mode
15
[DEBUG] scalar mode 15 took 0.160 s
[DEBUG] tensor mode 15 took 0.146 s
Mode 16/214 (k = 1.066e-61) took 0.31 s
Finished for this mode

Starting for this mode
16
[DEBUG] scalar mode 16 took 0.156 s
[DEBUG] tensor mode 16 took 0.141 s
Mode 17/214 (k = 1.322e-61) took 0.30 s
Finished for this mode

Starting for this mode
17
[DEBUG] scalar mode 17 took 0.159 s
[DEBUG] tensor mode 17 took 0.143 s
Mode 18/214 (k = 1.639e-61) took 0.30 s
Finished for this mode

Starting for this mode
18
[DEBUG] scalar mode 18 took 0.158 s
[DEBUG] tensor mode 18 took 0.145 s
Mode 19/214 (k = 2.033e-61) took 0.30 s
Finished for this mode

Starting for this mode
19
[DEBUG] scalar mode 19 took 0.159 s
[DEBUG] tensor mode 19 took 0.143 s
Mode 20/214 (k = 2.521e-61) took 0.30 s
Finished for this mode

Starting for this mode
20
[DEBUG] scalar mode 20 took 0.159 s
[DEBUG] tensor mode 20 took 

[DEBUG] tensor mode 65 took 0.133 s
Mode 66/214 (k = 5.052e-57) took 0.28 s
Finished for this mode

Starting for this mode
66
[DEBUG] scalar mode 66 took 0.146 s
[DEBUG] tensor mode 66 took 0.128 s
Mode 67/214 (k = 6.266e-57) took 0.27 s
Finished for this mode

Starting for this mode
67
[DEBUG] scalar mode 67 took 0.145 s
[DEBUG] tensor mode 67 took 0.133 s
Mode 68/214 (k = 7.771e-57) took 0.28 s
Finished for this mode

Starting for this mode
68
[DEBUG] scalar mode 68 took 0.145 s
[DEBUG] tensor mode 68 took 0.133 s
Mode 69/214 (k = 9.639e-57) took 0.28 s
Finished for this mode

Starting for this mode
69
[DEBUG] scalar mode 69 took 0.147 s
[DEBUG] tensor mode 69 took 0.133 s
Mode 70/214 (k = 1.195e-56) took 0.28 s
Finished for this mode

Starting for this mode
70
[DEBUG] scalar mode 70 took 0.145 s
[DEBUG] tensor mode 70 took 0.127 s
Mode 71/214 (k = 1.483e-56) took 0.27 s
Finished for this mode

Starting for this mode
71
[DEBUG] scalar mode 71 took 0.145 s
[DEBUG] tensor mode 71 took 

[DEBUG] scalar mode 117 took 0.135 s
[DEBUG] tensor mode 117 took 0.119 s
Mode 118/214 (k = 3.685e-52) took 0.25 s
Finished for this mode

Starting for this mode
118
[DEBUG] scalar mode 118 took 0.136 s
[DEBUG] tensor mode 118 took 0.122 s
Mode 119/214 (k = 4.570e-52) took 0.26 s
Finished for this mode

Starting for this mode
119
[DEBUG] scalar mode 119 took 0.135 s
[DEBUG] tensor mode 119 took 0.117 s
Mode 120/214 (k = 5.668e-52) took 0.25 s
Finished for this mode

Starting for this mode
120
[DEBUG] scalar mode 120 took 0.134 s
[DEBUG] tensor mode 120 took 0.120 s
Mode 121/214 (k = 7.030e-52) took 0.25 s
Finished for this mode

Starting for this mode
121
[DEBUG] scalar mode 121 took 0.134 s
[DEBUG] tensor mode 121 took 0.120 s
Mode 122/214 (k = 8.719e-52) took 0.26 s
Finished for this mode

Starting for this mode
122
[DEBUG] scalar mode 122 took 0.134 s
[DEBUG] tensor mode 122 took 0.118 s
Mode 123/214 (k = 1.081e-51) took 0.25 s
Finished for this mode

Starting for this mode
123
[DEB

[DEBUG] tensor mode 167 took 0.108 s
Mode 168/214 (k = 1.747e-47) took 0.23 s
Finished for this mode

Starting for this mode
168
[DEBUG] scalar mode 168 took 0.127 s
[DEBUG] tensor mode 168 took 0.108 s
Mode 169/214 (k = 2.167e-47) took 0.24 s
Finished for this mode

Starting for this mode
169
[DEBUG] scalar mode 169 took 0.128 s
[DEBUG] tensor mode 169 took 0.107 s
Mode 170/214 (k = 2.687e-47) took 0.24 s
Finished for this mode

Starting for this mode
170
[DEBUG] scalar mode 170 took 0.124 s
[DEBUG] tensor mode 170 took 0.107 s
Mode 171/214 (k = 3.333e-47) took 0.23 s
Finished for this mode

Starting for this mode
171
[DEBUG] scalar mode 171 took 0.124 s
[DEBUG] tensor mode 171 took 0.105 s
Mode 172/214 (k = 4.134e-47) took 0.23 s
Finished for this mode

Starting for this mode
172
[DEBUG] scalar mode 172 took 0.124 s
[DEBUG] tensor mode 172 took 0.108 s
Mode 173/214 (k = 5.127e-47) took 0.23 s
Finished for this mode

Starting for this mode
173
[DEBUG] scalar mode 173 took 0.124 s
[DEB

In [2]:
#This will run inside the notebook
#%prun runs profiler on the code
#-s time sorts results by total time spent in each function
#limits output to top 20 function
#call teh function you wanna use which for me is spectrum and requires the arg

#line below is what was running before
# %prun -s cumtime run_neqs8_models()